# SAM-Audio Small · 사전학습 모델 평가
추가 학습 없이 validation 혼합음에서 악기별 소리를 추출하고 기존 SDR 기준으로 평가합니다. train은 다운로드하지 않습니다.

**준비:** GPU 런타임, GCP 권한, Colab Secrets의 `WANDB_API_KEY`와 `HF_TOKEN`(각 노트북 액세스 켜기).
[Hugging Face 모델](https://huggingface.co/facebook/sam-audio-small)에서 접근 승인을 받은 계정의 읽기 토큰을 사용하세요.
BF16을 지원하는 GPU가 필요합니다. T4 등에서 지원하지 않으면 더 큰 GPU를 사용하거나 YAML `dtype: float32`로 바꾸세요(메모리 증가). FP16으로 자동 전환하지 않습니다.

1~6번 셀을 순서대로 실행하세요. 기본은 전체 validation입니다. 처음 연결 확인에는 YAML `parameters.max_scenes: 1`을 사용하세요(전체 성능이 아닌 진단 결과).
실험 설정과 프롬프트는 3번 셀 실행 전에 YAML에서 바꾸세요. 변경 후에는 새 실험을 시작합니다.
공식 코드 기반이며 이 저장소에서는 실제 Colab GPU·가중치 실행을 아직 검증하지 않았습니다.

## 1번 셀 · GitHub 코드와 SAM-Audio 설치
공식 SAM-Audio 커밋을 고정합니다. 새 GPU 런타임에서 시작하세요. 설치 중 이미 import한 패키지가 있다면 런타임을 재시작한 후 1번부터 실행하세요.

In [ ]:
REPOSITORY = "c-peace/ensemble_banchmark_pipeline"
REF = "exp/peace/Sam_audio"
CONFIG_FILE = "configs/experiments/sam_audio_pretrained.yaml"

import os
import re
import subprocess
import sys
import tempfile
from pathlib import Path


def secret(name):
    if os.environ.get(name):
        return os.environ[name]
    try:
        from google.colab import userdata
    except ImportError:
        return None
    try:
        return userdata.get(name)
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        return None


if not re.fullmatch(r"[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+", REPOSITORY):
    raise ValueError("REPOSITORY는 GitHub의 owner/repository 형식입니다.")
if not REF or REF.startswith("-") or not re.fullmatch(r"[A-Za-z0-9_./-]+", REF):
    raise ValueError("REF에 브랜치, 태그 또는 커밋을 입력하세요.")
CODE_ROOT = Path("/content/ensemble-code")
url = f"https://github.com/{REPOSITORY}.git"
with tempfile.TemporaryDirectory() as temporary:
    askpass = Path(temporary) / "askpass.py"
    askpass.write_text("#!/usr/bin/env python3\nimport os, sys\nprint('x-access-token' if 'username' in sys.argv[1].lower() else os.environ.get('GITHUB_TOKEN', ''))\n")
    askpass.chmod(0o700)
    git_env = dict(os.environ, GIT_TERMINAL_PROMPT="0", GIT_ASKPASS=str(askpass),
                   GITHUB_TOKEN=secret("GITHUB_TOKEN") or "")

    def git(*args):
        result = subprocess.run(["git", *args], env=git_env, capture_output=True, text=True)
        if result.returncode:
            raise RuntimeError("GitHub 다운로드 실패: 저장소·REF·토큰 권한을 확인하세요.")
        return result.stdout.strip()

    if not CODE_ROOT.exists():
        # 실패한 다운로드가 다음 실행에 남지 않도록 임시 폴더에서 준비합니다.
        checkout = Path(temporary) / "checkout"
        git("init", "--quiet", str(checkout))
        git("-C", str(checkout), "remote", "add", "origin", url)
        git("-C", str(checkout), "fetch", "--quiet", "--depth", "1", "origin", REF)
        git("-C", str(checkout), "checkout", "--quiet", "--detach", "FETCH_HEAD")
        import shutil
        shutil.move(str(checkout), str(CODE_ROOT))
    else:
        if git("-C", str(CODE_ROOT), "remote", "get-url", "origin") != url:
            raise ValueError("다른 저장소가 이미 있습니다. 새 Colab 런타임에서 실행하세요.")
        if git("-C", str(CODE_ROOT), "status", "--porcelain"):
            raise ValueError("가져온 코드가 수정되었습니다. 변경을 저장하고 새 런타임에서 실행하세요.")
        git("-C", str(CODE_ROOT), "fetch", "--quiet", "--depth", "1", "origin", REF)
        if git("-C", str(CODE_ROOT), "rev-parse", "HEAD") != git("-C", str(CODE_ROOT), "rev-parse", "FETCH_HEAD"):
            raise ValueError("GitHub 코드가 변경되었습니다. 새 런타임에서 다시 시작하세요.")
    CODE_COMMIT = git("-C", str(CODE_ROOT), "rev-parse", "HEAD")
os.chdir(CODE_ROOT)
SAM_COMMIT = "bb4c6999d2677c7402360e426afc01ddfad6dce0"
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", ".[colab]",
    f"sam_audio @ git+https://github.com/facebookresearch/sam-audio.git@{SAM_COMMIT}"], check=True)
print("코드 준비 완료:", CODE_COMMIT)

## 2번 셀 · validation만 다운로드
권한을 받은 Google 계정으로 인증하세요. 원본 음원은 변경하지 않습니다.

In [ ]:
from google.colab import auth
from ensemble_benchmark.cloud import download_data
from ensemble_benchmark.config import load_config

config = load_config(CODE_ROOT / CONFIG_FILE)
auth.authenticate_user(project_id=config['data']['gcp_project'])
DATA_ROOT = download_data(config['data']['gcs_uri'], '/content/ensemble-data',
    project=config['data']['gcp_project'], splits=('validation',))
VALID_ROOT = DATA_ROOT / 'validation'
print('Validation:', VALID_ROOT)

## 3번 셀 · 새 W&B 평가 실험 시작
이 셀을 다시 실행하면 새 run이 만들어집니다. 프롬프트는 목표 악기 이름만 사용합니다. 정답 음원의 시간 정보나 내용으로 프롬프트를 고르지 않습니다.
`Bass`는 double bass, `Horn`은 french horn, `untunedpercussion`은 unpitched percussion으로 구분했습니다. 프롬프트 성능은 아직 검증되지 않았으며 YAML에서 수정할 수 있습니다.

In [ ]:
import json
import uuid
import yaml
import math
import copy
import wandb

p = config['parameters']
if p['model_id'] != 'facebook/sam-audio-small':
    raise ValueError('이 노트북은 sam-audio-small 평가용입니다.')
for key in ('chunk_seconds', 'context_seconds'):
    if not isinstance(p[key], (int, float)) or not math.isfinite(p[key]) or p[key] < 0:
        raise ValueError(f'{key} must be finite and nonnegative')
if p['chunk_seconds'] <= 0 or p['context_seconds'] >= p['chunk_seconds']:
    raise ValueError('0 <= context_seconds < chunk_seconds 이어야 합니다.')
if p['dtype'] not in ('bfloat16', 'float32'):
    raise ValueError('dtype은 bfloat16 또는 float32입니다.')
if p['max_scenes'] is not None and (type(p['max_scenes']) is not int or p['max_scenes'] < 1):
    raise ValueError('max_scenes는 null 또는 양의 정수입니다.')
if type(p['seed']) is not int or not 0 <= p['seed'] < 2**32:
    raise ValueError('seed must be an integer in [0, 2**32)')
if not isinstance(p['prompts'], dict) or not all(isinstance(v, str) and v.strip() for v in p['prompts'].values()):
    raise ValueError('악기별 텍스트 프롬프트를 입력하세요.')
if (config['data']['smoke'] or not config['evaluation']['enabled']
        or config['evaluation']['max_seconds'] is not None or config['evaluation']['include_metrics']):
    raise ValueError('이 노트북은 길이 제한 없는 SDR 평가입니다. 장면 제한은 parameters.max_scenes를 사용하세요.')
experiment = config['experiment']
if wandb.run is not None:
    wandb.run.finish()
if experiment['wandb_mode'] == 'online':
    wandb.login(key=secret('WANDB_API_KEY'))
RUN_ID = f"{experiment['name']}-{uuid.uuid4().hex[:8]}"
RUN_DIR = Path('/content/ensemble-runs') / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)
CONFIG_SNAPSHOT = RUN_DIR / 'config.yaml'
CONFIG_SNAPSHOT.write_text(yaml.safe_dump(config, allow_unicode=True, sort_keys=False))
run = wandb.init(project=experiment['wandb_project'], entity=experiment['wandb_entity'],
    name=RUN_ID, id=RUN_ID, job_type='pretrained-evaluation',
    mode=experiment['wandb_mode'], dir=str(RUN_DIR),
    config={**config, 'github_commit': CODE_COMMIT, 'sam_audio_commit': SAM_COMMIT})
REPORT_PATH = None
MODEL_INFO = None
print('평가 실험:', RUN_ID, run.url)

## 4번 셀 · 사전학습 모델 로딩
학습·optimizer 없이 `eval()`과 `inference_mode()`를 사용합니다. 사용하지 않는 구간 예측·재순위 모델은 로드하지 않습니다.
기본 BF16은 GPU 지원 여부를 확인합니다. 모델 로딩에는 GPU 외에 충분한 시스템 RAM도 필요합니다. 403 오류라면 HF_TOKEN 계정의 모델 접근 승인을 확인하세요.

In [ ]:
import gc
import torch
from huggingface_hub import login, snapshot_download
from sam_audio import SAMAudio, SAMAudioProcessor

if config != yaml.safe_load(CONFIG_SNAPSHOT.read_text()):
    raise ValueError('설정이 변경되었습니다. 3번 셀부터 새 실험을 시작하세요.')
MODEL_INFO = None
if not torch.cuda.is_available():
    raise RuntimeError('GPU 런타임으로 변경하세요.')
DTYPE = getattr(torch, p['dtype'])
if DTYPE == torch.bfloat16 and torch.cuda.get_device_capability()[0] < 8:
    raise RuntimeError('이 GPU는 BF16을 기본 지원하지 않습니다. 지원 GPU 또는 float32 설정이 필요합니다.')
hf_token = secret('HF_TOKEN')
if not hf_token:
    raise ValueError('Colab Secrets에 HF_TOKEN을 등록하고 노트북 액세스를 켜세요.')
login(token=hf_token, add_to_git_credential=False)
del hf_token
if 'model' in globals():
    del model
    gc.collect()
    torch.cuda.empty_cache()
MODEL_DIR = Path(snapshot_download(repo_id=p['model_id'], revision=p['model_revision'],
    allow_patterns=['config.json', 'checkpoint.pt']))
processor = SAMAudioProcessor.from_pretrained(str(MODEL_DIR))
model = SAMAudio.from_pretrained(str(MODEL_DIR), span_predictor=None,
    text_ranker=None, visual_ranker=None).eval()
model.requires_grad_(False)
model = model.to(device='cuda', dtype=DTYPE)
MODEL_INFO = {'model_id': p['model_id'], 'revision': MODEL_DIR.name,
    'dtype': p['dtype'], 'gpu': torch.cuda.get_device_name(),
    'predict_spans': False, 'reranking_candidates': 1, 'ode_steps': 16,
    'sample_rate': processor.audio_sampling_rate, 'sam_audio_commit': SAM_COMMIT}
MODEL_CONFIG = copy.deepcopy(config)
run.summary['model'] = MODEL_INFO
print(MODEL_INFO)

## 5번 셀 · 악기별 추론과 SDR 평가
전체 음원을 짧은 구간으로 처리합니다. 각 구간 양옆 context를 함께 추론하고 중심 부분만 이어 붙입니다. 이는 공식 long-form multi-diffusion 재현이 아닌 자체 구간 처리 방식이며 경계와 문맥 손실이 점수에 영향을 줄 수 있습니다.
모델 입력은 processor의 샘플레이트로 변환하고 출력은 기존 평가 기준인 44.1kHz mono로 복원합니다. 정답은 추론에 넣지 않습니다.
전체 평가는 오래 걸릴 수 있습니다. max_scenes가 설정되면 전체 점수 대신 diagnostic으로 기록합니다. 오류 후 이 셀을 재실행하면 해당 실험의 추론을 처음부터 다시 수행합니다.

In [ ]:
import hashlib
import time
import numpy as np
import soundfile as sf
import torchaudio
from tqdm.auto import tqdm
from ensemble_benchmark.scoring import score_dataset

if (config != yaml.safe_load(CONFIG_SNAPSHOT.read_text())
        or MODEL_INFO is None or MODEL_CONFIG != config):
    raise ValueError('3~4번 셀에서 현재 설정으로 실험과 모델을 준비하세요.')
REPORT_PATH = None
for key in ('evaluation',):
    if key in run.summary.keys():
        del run.summary[key]


def safe_path(root, relative):
    path = (Path(root) / relative).resolve()
    if not path.is_relative_to(Path(root).resolve()):
        raise ValueError(f'허용되지 않는 경로: {relative}')
    return path


def separate_audio(audio, rate, prompt):
    # audio: CPU float32 mono tensor. Reference stems are never passed here.
    n = audio.numel()
    chunk = max(1, round(p['chunk_seconds'] * rate))
    context = round(p['context_seconds'] * rate)
    output = torch.empty(n, dtype=torch.float32)
    for start in range(0, n, chunk):
        end = min(start + chunk, n)
        left, right = max(0, start - context), min(n, end + context)
        segment = torchaudio.functional.resample(audio[left:right].unsqueeze(0),
            rate, processor.audio_sampling_rate)
        batch = processor(audios=[segment], descriptions=[prompt]).to('cuda')
        batch.audios = batch.audios.to(dtype=DTYPE)
        with torch.inference_mode(), torch.autocast('cuda', dtype=DTYPE,
                enabled=DTYPE == torch.bfloat16):
            result = model.separate(batch, predict_spans=False, reranking_candidates=1,
                ode_opt={'method': 'midpoint', 'options': {'step_size': 1 / 16}})
        # Official separate() returns one tensor per batch item in a list.
        target = result.target[0].detach().float().cpu().reshape(-1)
        if target.numel() < segment.shape[-1] or not torch.isfinite(target).all():
            raise ValueError('SAM-Audio 출력 길이가 부족하거나 유한하지 않습니다.')
        target = target[:segment.shape[-1]]
        restored = torchaudio.functional.resample(target, processor.audio_sampling_rate, rate)
        if restored.numel() < right - left:
            raise ValueError('리샘플링 출력 길이가 부족합니다.')
        output[start:end] = restored[start-left:end-left]
        del result, batch
    return output.numpy()


manifest_path = VALID_ROOT / 'manifest.json'
manifest = json.loads(manifest_path.read_text())
scenes = manifest['scenes']
if not scenes or any(s.get('split') != 'validation' for s in scenes):
    raise ValueError('validation manifest를 확인하세요.')
if len({s['scene_id'] for s in scenes}) != len(scenes):
    raise ValueError('중복 scene_id입니다.')
if not manifest.get('complete'):
    raise ValueError('완전한 validation manifest가 필요합니다.')
expected = manifest.get('expected_scene_ids')
if not expected or len(set(expected)) != len(expected) or set(expected) != {s['scene_id'] for s in scenes}:
    raise ValueError('validation 장면 목록이 expected_scene_ids와 다릅니다.')
DIAGNOSTIC = p['max_scenes'] is not None
selected = scenes if not DIAGNOSTIC else scenes[:p['max_scenes']]
eval_manifest = copy.deepcopy(manifest)
eval_manifest['scenes'] = selected
eval_manifest['complete'] = not DIAGNOSTIC
PREDICTIONS = RUN_DIR / 'predictions'
for scene in selected:
    for target in scene['target_classes']:
        if target not in p['prompts']:
            raise ValueError(f'YAML에 {target} 프롬프트가 없습니다.')
        safe_path(PREDICTIONS, f"{scene['scene_id']}/{target}.wav")
    if set(scene['references']) != set(scene['target_classes']):
        raise ValueError('정답 악기와 target_classes가 다릅니다.')
    for relative in [scene['mixture'], *scene['references'].values()]:
        path = safe_path(VALID_ROOT, relative)
        info = sf.info(path)
        if (info.samplerate != 44100 or info.channels != 1
                or info.frames != scene['num_samples'] or scene['sample_rate'] != 44100):
            raise ValueError(f'오디오 형식이 manifest와 다릅니다: {relative}')
        with path.open('rb') as stream:
            digest = hashlib.file_digest(stream, 'sha256').hexdigest()
        if digest != scene['sha256'].get(relative):
            raise ValueError(f'오디오 체크섬 불일치: {relative}')

torch.manual_seed(p['seed'])
np.random.seed(p['seed'])
torch.cuda.reset_peak_memory_stats()
started = time.perf_counter()
total_targets = sum(len(s['target_classes']) for s in selected)
with tqdm(total=total_targets, desc='SAM-Audio Small', unit='target') as progress:
    for scene in selected:
        audio, rate = sf.read(safe_path(VALID_ROOT, scene['mixture']), dtype='float32')
        tensor = torch.from_numpy(audio)
        for target in scene['target_classes']:
            predicted = separate_audio(tensor, rate, p['prompts'][target])
            destination = safe_path(PREDICTIONS, f"{scene['scene_id']}/{target}.wav")
            destination.parent.mkdir(parents=True, exist_ok=True)
            sf.write(destination, predicted, rate, subtype='FLOAT')
            progress.update(1)
            run.log({'inference/completed_targets': progress.n})

torch.cuda.synchronize()
inference_seconds = time.perf_counter() - started
report = score_dataset(eval_manifest, VALID_ROOT, PREDICTIONS,
    split='validation', allow_partial=DIAGNOSTIC, show_progress=True)
report.update(diagnostic=DIAGNOSTIC, model=MODEL_INFO, prompts=p['prompts'],
    seed=p['seed'], chunk_seconds=p['chunk_seconds'], context_seconds=p['context_seconds'],
    manifest_sha256=hashlib.sha256(manifest_path.read_bytes()).hexdigest(),
    inference_seconds=inference_seconds,
    peak_gpu_allocated_gb=torch.cuda.max_memory_allocated() / 1024**3)
if DIAGNOSTIC:
    report['overall'] = None
report_path = RUN_DIR / 'evaluation.json'
report_path.write_text(json.dumps(report, indent=2, allow_nan=False))
run.summary['evaluation'] = report
metrics = {'inference/seconds': inference_seconds,
           'inference/peak_gpu_allocated_gb': report['peak_gpu_allocated_gb']}
prefix = 'diagnostic' if DIAGNOSTIC else 'validation'
if report['overall'] is not None:
    metrics[f'{prefix}/sdr'] = report['overall']
for dataset, values in report['datasets'].items():
    for instrument, value in values['instruments'].items():
        metrics[f'{prefix}/{dataset}/{instrument}/sdr'] = value
run.log(metrics)
REPORT_PATH = report_path
print('SDR 평가 완료:', REPORT_PATH)
print('전체 SDR:', report['overall'], '| diagnostic:', DIAGNOSTIC)

## 6번 셀 · W&B에 평가 결과 저장 후 종료
평가 JSON, 사용한 YAML, manifest, 실행 환경과 모델 revision을 저장합니다. 재배포가 필요 없는 사전학습 가중치는 업로드하지 않습니다. 추론 음원은 Colab의 run 폴더에 남으며 이 셀은 음원 전체를 업로드하지 않습니다. 학습을 하지 않으므로 train loss도 없습니다.

In [ ]:
import subprocess

if config != yaml.safe_load(CONFIG_SNAPSHOT.read_text()):
    raise ValueError('평가 후 설정이 변경되었습니다. 새 실험을 시작하세요.')
if REPORT_PATH is None or not Path(REPORT_PATH).is_file():
    raise ValueError('먼저 5번 셀의 평가를 완료하세요.')
(RUN_DIR / 'environment.txt').write_text(subprocess.check_output(
    [sys.executable, '-m', 'pip', 'freeze'], text=True))
(RUN_DIR / 'evaluation-manifest.json').write_text(json.dumps(eval_manifest, indent=2))
artifact = wandb.Artifact(RUN_ID, type='evaluation', metadata=MODEL_INFO)
for path in (REPORT_PATH, CONFIG_SNAPSHOT, RUN_DIR / 'environment.txt',
             RUN_DIR / 'evaluation-manifest.json'):
    artifact.add_file(str(path), name=path.name)
logged = run.log_artifact(artifact)
if experiment['wandb_mode'] == 'online':
    logged.wait()
run.finish()
print('W&B 평가 결과 저장 완료:', RUN_ID)